In [23]:
import torch
import torch.nn as nn 
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

In [12]:
class PriceDataset(Dataset):
    def __init__(self, img_path, image_paths, item_names, bullet_points, product_descriptions, values, units, prices, processor):
        self.dir = img_path
        self.image_paths = image_paths
        self.item_names = item_names
        self.bullet_points = bullet_points
        self.product_descriptions = product_descriptions
        self.values = values
        self.units = units
        self.prices = torch.tensor(prices, dtype=torch.float32)  # Changed to float32 for regression
        self.processor = processor
    
    def __len__(self):
        return len(self.image_paths)
    
    def _concatenate_text(self, idx):
        """Concatenate all text fields into one string"""
        text_parts = []
        
        # Add Item Name
        if pd.notna(self.item_names[idx]):
            text_parts.append(f"Item Name: {self.item_names[idx]}")
        
        # Add Bullet Points
        if pd.notna(self.bullet_points[idx]):
            text_parts.append(f"Bullet Points: {self.bullet_points[idx]}")
        
        # Add Product Description
        if pd.notna(self.product_descriptions[idx]):
            text_parts.append(f"Product Description: {self.product_descriptions[idx]}")
        
        # Add Value and Unit
        if pd.notna(self.values[idx]) and pd.notna(self.units[idx]):
            text_parts.append(f"Quantity: {self.values[idx]} {self.units[idx]}")
        
        return " ".join(text_parts)
    
    def __getitem__(self, idx):
        path = os.path.join(self.dir, str(self.image_paths[idx]) + '.jpg')
        img = Image.open(path).convert('RGB')
        
        # Concatenate all text fields
        full_text = self._concatenate_text(idx)
        
        inputs = self.processor(
            images=img, 
            text=full_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'price': self.prices[idx]
        }

In [15]:
class SMAPELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    
    def forward(self, pred, target):
        return torch.mean(2.0 * torch.abs(pred - target) / (torch.abs(pred) + torch.abs(target) + self.eps)) * 100

In [16]:
class BLIP2PricePredictor(nn.Module):
    def __init__(self, model_name="Salesforce/blip2-opt-2.7b"):
        super().__init__()
        self.blip2 = Blip2ForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        
        for param in self.blip2.parameters():
            param.requires_grad = False
        
        qformer_dim = self.blip2.config.qformer_config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(qformer_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )
    
    def forward(self, pixel_values, input_ids, attention_mask):
        vision_outputs = self.blip2.vision_model(pixel_values)
        image_embeds = vision_outputs[0]
        
        image_attn_mask = torch.ones(
            image_embeds.size()[:-1], 
            dtype=torch.long, 
            device=image_embeds.device
        )
        
        query_tokens = self.blip2.query_tokens.expand(image_embeds.shape[0], -1, -1)
        
        text_embeds = self.blip2.qformer.bert.embeddings(input_ids=input_ids)
        
        query_outputs = self.blip2.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attn_mask,
            return_dict=True,
        )
        
        qformer_out = query_outputs.last_hidden_state[:, 0, :]
        price_pred = self.regressor(qformer_out)
        return price_pred.squeeze(-1)


In [17]:
def apply_qlora(model, r=16, alpha=32):
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=[
            "query", "key", "value",
            "attention.self.query",
            "attention.self.key", 
            "attention.self.value",
            "crossattention.self.query",
            "crossattention.self.key",
            "crossattention.self.value"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    
    model.blip2.qformer = get_peft_model(model.blip2.qformer, lora_config)
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    return model

In [ ]:
def train(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda', log_file='training_log.csv'):
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=0.01
    )
    criterion = SMAPELoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    accum_steps = 4
    
    # Initialize CSV logging
    import csv
    import os
    
    # Create log file with headers if it doesn't exist
    if not os.path.exists(log_file):
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Epoch', 'Train_Loss', 'Val_Loss'])
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for i, batch in enumerate(train_loader):
            pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            prices = batch['price'].to(device)
            
            preds = model(pixel_values, input_ids, attention_mask)
            loss = criterion(preds.float(), prices) / accum_steps
            loss.backward()
            
            if (i + 1) % accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accum_steps
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                prices = batch['price'].to(device)
                
                preds = model(pixel_values, input_ids, attention_mask)
                val_loss += criterion(preds.float(), prices).item()
        
        # Calculate average losses
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        # Log to CSV file
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, avg_train_loss, avg_val_loss])
        
        scheduler.step()
        print(f"Epoch {epoch+1}: Train={avg_train_loss:.2f}% Val={avg_val_loss:.2f}%")
    
    print(f"Training log saved to: {log_file}")
    return model

In [ ]:
MODEL = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(MODEL)
TRAIN_IMG_DIR = '../images/train_part1'

train_df = pd.read_csv('../dataset/train_split/part1.csv')
val_df = pd.read_csv('../dataset/val_split/part1.csv') 

train_dl = DataLoader(
    PriceDataset(
        img_path = TRAIN_IMG_DIR,
        image_paths=train_df['sample_id'].values,
        item_names=train_df['Item Name'].values,
        bullet_points=train_df['Bullet Points'].values,
        product_descriptions=train_df['Product Description'].values,
        values=train_df['Value'].values,
        units=train_df['Unit'].values,
        prices=train_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=True,
    num_workers=4
)
val_dl = DataLoader(
    PriceDataset(
        img_path = TRAIN_IMG_DIR,
        image_paths=val_df['sample_id'].values,
        item_names=val_df['Item Name'].values,
        bullet_points=val_df['Bullet Points'].values,
        product_descriptions=val_df['Product Description'].values,
        values=val_df['Value'].values,
        units=val_df['Unit'].values,
        prices=val_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4
)    

model = BLIP2PricePredictor(MODEL)
model = apply_qlora(model, r=16, alpha=32)

model = train(model, train_dl, val_dl, epochs=10, lr=1e-4)
torch.save({
        'regressor': model.regressor.state_dict(),
        'qformer': model.blip2.qformer.state_dict(),
    }, 'model.pth')
    
print("Done!")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'pixel_values': tensor([[[1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
         [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
         [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
         ...,
         [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
         [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303],
         [1.9303, 1.9303, 1.9303,  ..., 1.9303, 1.9303, 1.9303]],

        [[2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
         [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
         [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
         ...,
         [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
         [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749],
         [2.0749, 2.0749, 2.0749,  ..., 2.0749, 2.0749, 2.0749]],

        [[2.1459, 2.1459, 2.1459,  ..., 2.1459, 2.1459, 2.1459],
         [2.1459, 2.1459, 2.1459,  ..., 2.1459, 2.1459, 2.1459],
         [2.1459, 2.1459, 2.1459,  ..., 2